In [71]:
import geopandas as gpd
import pandas as pd
from shapely import Point
from sklearn.preprocessing import MinMaxScaler

from src.features.geo import water_fraction, within_shape, extract_polygon
from src.preprocessing.preprocessing import construct_basin_location_mapper, \
    construct_basin_location_mapper_with_external, onehot_encode, split_binarize_encode

### Загрузка данных

#### Обучающий и тестовый датасет

In [2]:
train_oil_df = pd.read_csv("../data/train_oil.csv")
test_oil_df = pd.read_csv("../data/oil_test.csv")

#### Пропущенные координаты, бассейны и мэппинг бассейнов

In [3]:
missing_coordinates = pd.read_csv("../data/missing_coordinates.csv")
missing_coordinates = missing_coordinates.fillna("")

In [4]:
missing_basins = pd.read_csv("../data/missing_basins.csv")

In [5]:
basins_mapped = pd.read_csv("../data/basins_mapped.csv")

#### Гео-датасеты

In [6]:
land = gpd.read_file("../data/ne_50m_land.zip")['geometry']
land_10m = gpd.read_file("../data/ne_10m_land.zip")['geometry']
ocean_50m = gpd.read_file("../data/ne_50m_ocean.zip")['geometry']
ocean_110m = gpd.read_file("../data/ne_110m_ocean.zip")['geometry']
ne_50m_geography_marine_polys = gpd.read_file("../data/ne_50m_geography_marine_polys.zip")
lakes = gpd.read_file("../data/ne_50m_lakes.zip")['geometry']
geography_regions = gpd.read_file("../data/ne_50m_geography_regions_polys.zip")
rivers_lake_centerlines = gpd.read_file("../data/ne_50m_rivers_lake_centerlines.zip")['geometry']
major_islands = gpd.read_file("../data/ne_50m_coastline.zip")['geometry']
minor_islands = gpd.read_file("../data/ne_10m_minor_islands.zip")['geometry']

### Подготовка данных

In [7]:
train_str_cols = train_oil_df.select_dtypes(include='str').columns
test_str_cols = test_oil_df.select_dtypes(include='str').columns
train_oil_df[train_str_cols] = train_oil_df[train_str_cols].apply(lambda x: x.str.lower())
test_oil_df[test_str_cols] = test_oil_df[test_str_cols].apply(lambda x: x.str.lower())
train_oil_df.columns = [c.lower().replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_") for c in
                        train_oil_df.columns]

In [8]:
train_oil_df['region'] = train_oil_df['region'].fillna('unknown')
train_oil_df['country'] = train_oil_df['country'].fillna('unknown')

In [9]:
train_oil_df_filled = train_oil_df.merge(missing_coordinates[["field_name", "reservoir_unit", "latitude", "longitude"]],
                                         on=["field_name", "reservoir_unit"],
                                         how='left', suffixes=('', '_fill'))
train_oil_df_filled['latitude'] = train_oil_df_filled['latitude'].fillna(train_oil_df_filled['latitude_fill'])
train_oil_df_filled['longitude'] = train_oil_df_filled['longitude'].fillna(train_oil_df_filled['longitude_fill'])

In [10]:
train_oil_df_filled = train_oil_df_filled.merge(missing_basins[["basin_name", "field_name", "reservoir_unit"]],
                                                on=["field_name", "reservoir_unit"],
                                                how='left', suffixes=('', '_fill'))
train_oil_df_filled['basin_name'] = train_oil_df_filled['basin_name'].fillna(train_oil_df_filled['basin_name_fill'])

In [11]:
train_oil_df_filled.drop(columns=['latitude_fill', 'longitude_fill'], inplace=True)
train_oil_df_filled.drop(columns=['basin_name_fill'], inplace=True)

#### Гео датасет

In [12]:
gdf = gpd.GeoDataFrame(
    train_oil_df_filled,
    geometry=gpd.points_from_xy(train_oil_df_filled.longitude, train_oil_df_filled.latitude),
    crs="EPSG:4326"
)

#### Формирование подмножеств геопризнаков

In [13]:
bays = ne_50m_geography_marine_polys[ne_50m_geography_marine_polys.featurecla == 'bay']['geometry']
gulfs = ne_50m_geography_marine_polys[ne_50m_geography_marine_polys.featurecla == 'gulf']['geometry']
straits = ne_50m_geography_marine_polys[ne_50m_geography_marine_polys.featurecla == 'strait']['geometry']
channels = ne_50m_geography_marine_polys[ne_50m_geography_marine_polys.featurecla == 'channel']['geometry']
sounds = ne_50m_geography_marine_polys[ne_50m_geography_marine_polys.featurecla == 'sound']['geometry']
rivers = ne_50m_geography_marine_polys[ne_50m_geography_marine_polys.featurecla == 'river']['geometry']
isthmuses = geography_regions[geography_regions.FEATURECLA == 'Isthmus']['geometry']
deltas = geography_regions[geography_regions.FEATURECLA == 'Delta']['geometry']
coasts = geography_regions[geography_regions.FEATURECLA == 'Coast']['geometry']
lake_regions = geography_regions[geography_regions.FEATURECLA == 'Lake']['geometry']
islands = geography_regions[geography_regions.FEATURECLA.isin(['Island group', 'Island'])]['geometry']

In [14]:
bunyu_coords = Point(117.833, 3.5)
bunyu_island = gpd.GeoSeries(
    [extract_polygon(land_10m[land_10m.geometry.contains(bunyu_coords)].iloc[0], bunyu_coords)],
    crs="EPSG:4326"
)

In [15]:
all_lakes = gpd.GeoDataFrame(
    geometry=pd.concat([lakes, rivers_lake_centerlines, lake_regions], ignore_index=True),
    crs="EPSG:4326").dissolve()

In [16]:
all_islands = gpd.GeoDataFrame(
    geometry=pd.concat([islands, major_islands, minor_islands, bunyu_island], ignore_index=True),
    crs="EPSG:4326").dissolve()

### Подготовка признаков

#### Корректировка названия стран

In [17]:
gdf.loc[gdf['country'] == 'norway /uk', 'country'] = 'uk /norway'

#### Корректировка координат Berri Hanifa

In [18]:
berri_hanifa_coords = (27.11, 49.6389)
berri_hanifa_idx = (gdf["field_name"] == "berri") & (gdf["reservoir_unit"] == "hanifa")
gdf.loc[berri_hanifa_idx, 'latitude'] = berri_hanifa_coords[0]
gdf.loc[berri_hanifa_idx, 'longitude'] = berri_hanifa_coords[1]

#### Процент воды в радиусе 2км

In [19]:
gdf['water_pct'] = gdf.apply(lambda row: water_fraction(point_lat=row.latitude,
                                                        point_lon=row.longitude,
                                                        water_shapes=ocean_50m,
                                                        radius_km=2), axis=1)

#### Гео-признаки

In [20]:
gdf['is_in_water'] = gdf.apply(lambda row: within_shape(lat=row.latitude, lon=row.longitude, gdf=ocean_50m), axis=1)
gdf['is_on_island'] = gdf.apply(lambda row: within_shape(lat=row.latitude, lon=row.longitude, gdf=all_islands), axis=1)
gdf['is_in_gulf'] = gdf.apply(lambda row: within_shape(lat=row.latitude, lon=row.longitude, gdf=gulfs), axis=1)
gdf['is_in_strait'] = gdf.apply(lambda row: within_shape(lat=row.latitude, lon=row.longitude, gdf=straits), axis=1)
gdf['is_in_delta'] = gdf.apply(lambda row: within_shape(lat=row.latitude, lon=row.longitude, gdf=deltas), axis=1)
gdf['is_in_bay'] = gdf.apply(lambda row: within_shape(lat=row.latitude, lon=row.longitude, gdf=bays), axis=1)

#### Извлечение отметок onshore/offshore, lake из названий местрождения и бассейна

In [21]:
names_concat = gdf['field_name'] + gdf['reservoir_unit'] + gdf['basin_name']
gdf['ref_onshore'] = names_concat.str.contains('onshore')
gdf['ref_offshore'] = names_concat.str.contains('offshore')
gdf['ref_lake'] = names_concat.str.contains("lake")

#### Метки локаций бассейнов (расчётные и внешние)

In [22]:
basin_location = pd.crosstab(gdf['basin_name'], gdf['onshore_offshore']).drop('unknown', errors='ignore').reset_index()

loc_mask = ((basin_location['onshore'] > 0) &
            (basin_location['offshore'] > 0))

basin_location.loc[loc_mask, 'onshore-offshore-calc'] = (
        basin_location.loc[loc_mask, 'onshore'] +
        basin_location.loc[loc_mask, 'offshore'] +
        basin_location.loc[loc_mask, 'onshore-offshore'])

basin_location['onshore-offshore-calc'] = basin_location['onshore-offshore-calc'].fillna(0).astype(int)

loc_mask_2 = ((basin_location['onshore'] == 0) &
              (basin_location['offshore'] == 0) &
              (basin_location['onshore-offshore'] > 0))

basin_location.loc[loc_mask_2, 'onshore-offshore-calc'] = basin_location.loc[loc_mask_2, 'onshore-offshore']
inferred_location_map = construct_basin_location_mapper(basin_location)

In [23]:
basins_mapped = basins_mapped.drop(basins_mapped[basins_mapped['basin_name'] == 'unknown'].index).reset_index(
    drop=True).to_dict("records")
external_location_map = {b['basin_name']: b['location'] for b in basins_mapped}
basin_location_map_combined = construct_basin_location_mapper_with_external(basin_location, external_location_map)

In [24]:
gdf['basin_location_inferred'] = gdf['basin_name'].map(inferred_location_map).fillna("unknown")
gdf['basin_location_external'] = gdf['basin_name'].map(basin_location_map_combined).fillna('unknown')

### Разбивка и кодировка композитных полей 

In [72]:
country_features, country_binarizer = onehot_encode(gdf, 'country')
region_features, region_binarizer = onehot_encode(gdf, 'region')
basin_name_features, basin_name_binarizer = onehot_encode(gdf, 'basin_name')
tectonic_regime_features, tectonic_regime_binarizer = split_binarize_encode(gdf, 'tectonic_regime')
operator_company_features, operator_company_binarizer = onehot_encode(gdf, 'operator_company')
hydrocarbon_type_features, hydrocarbon_type_binarizer = onehot_encode(gdf, 'hydrocarbon_type')
structural_setting_features, structural_setting_binarizer = split_binarize_encode(gdf, 'structural_setting')
reservoir_period_features, reservoir_period_binarizer = onehot_encode(gdf, 'reservoir_period')
lithology_features, lithology_binarizer = onehot_encode(gdf, 'lithology')
basin_location_inferred_features, basin_location_inferred_binarizer = onehot_encode(gdf, 'basin_location_inferred')
basin_location_external_features, basin_location_external_binarizer = onehot_encode(gdf, 'basin_location_external')

In [73]:
numeric_cols = ['depth', 'thickness_gross_average_ft', 'thickness_net_pay_average_ft', 'porosity', 'permeability']
gdf_numeric = gdf[numeric_cols].copy()
numeric_scaler = MinMaxScaler()
gdf_scaled = gdf_numeric.copy()
gdf_scaled[numeric_cols] = numeric_scaler.fit_transform(gdf_numeric[numeric_cols])

In [81]:
binary_cols = ['is_in_water', 'is_on_island', 'is_in_gulf', 'is_in_strait', 'is_in_delta', 'is_in_bay', 'ref_onshore',
               'ref_offshore', 'ref_lake']
gdf_binary_features = gdf[binary_cols].astype(float).copy()

In [82]:
pd.concat([
    country_features,
    region_features,
    basin_name_features,
    tectonic_regime_features,
    operator_company_features,
    hydrocarbon_type_features,
    structural_setting_features,
    reservoir_period_features,
    lithology_features,
    basin_location_inferred_features,
    basin_location_external_features,
    gdf_scaled,
    gdf_binary_features
], axis=1)

,country_afghanistan,country_algeria,country_angola,country_australia,country_brazil,country_canada,country_china,country_colombia,country_denmark,country_egypt,...,permeability,is_in_water,is_on_island,is_in_gulf,is_in_strait,is_in_delta,is_in_bay,ref_onshore,ref_offshore,ref_lake
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.003999,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.046665,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.054665,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.099999,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.187599,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
304,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.013332,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
305,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.075465,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
306,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000012,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
307,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.299999,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
